# 03 · Normalization and Highly Variable Genes

**Goal**: Normalize gene expression data, apply log transformation, and identify highly variable genes (HVGs) for downstream analysis.

**Input**: results/adata/02_qc.h5ad

**Output**: results/adata/03_normalized.h5ad, normalization figures, and summary tables

**Note**: Original raw counts are preserved before normalization. All genes are retained, and QC filtering was not applied in Step 2.

## 1 · Imports and reproducible paths

In [ ]:
from pathlib import Path
from importlib.metadata import version
import sys
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse
import matplotlib.pyplot as plt


def setup_normalization_paths():
    cwd = Path.cwd().resolve()
    root = next((p for p in (cwd, *cwd.parents) if (p / "data").is_dir() and (p / "results").is_dir()), None)
    if root is None:
        raise FileNotFoundError("Run inside visium_hd_tutorial or its notebooks directory")
    input_path = root / "results" / "adata" / "02_qc.h5ad"
    output_path = root / "results" / "adata" / "03_normalized.h5ad"
    figures = root / "results" / "figures"
    tables = root / "results" / "tables"
    for directory in (output_path.parent, figures, tables):
        directory.mkdir(parents=True, exist_ok=True)
    assert input_path.is_file(), f"Missing input: {input_path}. Run 02_quality_control.ipynb first."
    return root, input_path, output_path, figures, tables


ROOT, INPUT, OUTPUT, FIGURES, TABLES = setup_normalization_paths()
print("Python:", sys.executable)
print("Scanpy:", version("scanpy"))
print("Project root:", ROOT)
print("Input:", INPUT.relative_to(ROOT))

## 2 · Load QC AnnData and validate raw counts

Load the QC-processed AnnData file from Step 2 and verify that the raw gene expression counts, spatial coordinates, and QC metrics are valid before normalization.

In [ ]:
def load_and_validate_qc(input_path):
    adata = sc.read_h5ad(input_path)
    assert sparse.issparse(adata.X), "Expected a sparse count matrix"
    assert adata.obs_names.is_unique and adata.var_names.is_unique
    assert "spatial" in adata.obsm and adata.obsm["spatial"].shape == (adata.n_obs, 2)
    assert "total_counts" in adata.obs and "n_genes_by_counts" in adata.obs
    assert adata.n_obs > 0 and adata.n_vars > 0
    assert np.isfinite(adata.X.data).all()
    assert np.all(adata.X.data >= 0), "Counts cannot be negative"
    assert np.allclose(adata.X.data, np.rint(adata.X.data)), "Input is not integer-like raw counts"
    return adata


adata = load_and_validate_qc(INPUT)
print(adata)
print("QC filtering status:", adata.uns.get("qc_filtering", "Check Step 02 metadata"))
print("Zero-count bins:", int(np.sum(np.asarray(adata.X.sum(axis=1)).ravel() == 0)))

## 3 · Preserve raw counts and calculate normalization factors

Preserve the original gene expression counts and calculate total counts for each spatial bin. Set the normalization target and the number of highly variable genes (HVGs) for subsequent analysis.

In [ ]:
def preserve_raw_counts(adata):
    # Preserve the full raw count matrix (not normalized or log-transformed).
    adata.layers["counts"] = adata.X.copy()
    raw_totals = np.asarray(adata.layers["counts"].sum(axis=1)).ravel()
    adata.obs["total_counts_before_normalization"] = raw_totals
    target_sum = 1e4
    n_top_genes = min(2000, adata.n_vars)
    return raw_totals, target_sum, n_top_genes


raw_totals, TARGET_SUM, N_TOP_GENES = preserve_raw_counts(adata)
print("Total counts before normalization (median):", float(np.median(raw_totals)))
print("Normalization target:", TARGET_SUM)
print("HVG target:", N_TOP_GENES)

## 4 · Library-size normalization and log1p

Normalize gene expression counts across spatial bins to a common total of 10,000, then apply log1p transformation to reduce differences in expression magnitude while preserving the original raw counts.

In [ ]:
def normalize_and_log1p(adata, raw_totals, target_sum):
    sc.pp.normalize_total(adata, target_sum=target_sum)
    normalized_totals = np.asarray(adata.X.sum(axis=1)).ravel()
    positive = raw_totals > 0
    assert np.allclose(normalized_totals[positive], target_sum, rtol=1e-4)
    assert np.all(normalized_totals[~positive] == 0)
    sc.pp.log1p(adata)
    assert sparse.issparse(adata.X), "Expected log-normalized matrix to remain sparse"
    assert np.isfinite(adata.X.data).all()
    return normalized_totals, positive


normalized_totals, positive = normalize_and_log1p(adata, raw_totals, TARGET_SUM)
print("Normalized and log1p-transformed:", adata.shape)
print("Raw counts preserved in adata.layers['counts']:", adata.layers["counts"].shape)

## 5 · Highly variable gene selection

Identify highly variable genes (HVGs) using the Seurat method to select genes with high expression variability for downstream analysis. All genes are retained, and the selected HVGs are saved as a CSV file.

In [ ]:
def select_and_save_hvgs(adata, n_top_genes, tables, root):
    sc.pp.highly_variable_genes(
        adata,
        flavor="seurat",
        n_top_genes=n_top_genes,
        subset=False,
        inplace=True,
    )
    assert "highly_variable" in adata.var
    n_hvg = int(adata.var["highly_variable"].sum())
    assert n_hvg > 0
    hvg_table = adata.var.loc[adata.var["highly_variable"], ["highly_variable", "means", "dispersions", "dispersions_norm"]].copy()
    hvg_table.index.name = "gene"
    hvg_path = tables / "03_highly_variable_genes.csv"
    hvg_table.to_csv(hvg_path)
    return n_hvg, hvg_path


n_hvg, hvg_path = select_and_save_hvgs(adata, N_TOP_GENES, TABLES, ROOT)
print("HVGs selected:", n_hvg)
print("All genes retained:", adata.n_vars)
print("Saved:", hvg_path.relative_to(ROOT))

## 6 · Visualize normalization results

Compare the distributions of total gene counts per spatial bin before and after normalization to verify that differences in library size have been reduced.

In [ ]:
def plot_library_size_normalization(raw_totals, normalized_totals, positive, figures):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    # Before normalization
    axes[0].hist(np.log1p(raw_totals), bins=60)
    axes[0].set(
        title="Raw library sizes",
        xlabel="log1p(total UMI per bin)",
        ylabel="Number of bins",
    )

    # After normalization: avoid automatic bins for near-constant values.
    values = np.asarray(normalized_totals[positive], dtype=float).ravel()
    values = values[np.isfinite(values)]

    if values.size == 0:
        axes[1].text(
            0.5, 0.5, "No nonzero bins",
            ha="center", va="center",
            transform=axes[1].transAxes,
        )
    elif np.ptp(values) < max(1e-6, abs(np.mean(values)) * 1e-8):
        axes[1].hist(
            values,
            bins=np.linspace(np.mean(values) - 1, np.mean(values) + 1, 21),
        )
    else:
        axes[1].hist(values, bins=40)

    axes[1].set(
        title="After library-size normalization (nonzero bins)",
        xlabel="Total normalized counts",
        ylabel="Number of bins",
    )

    fig.tight_layout()
    fig.savefig(
        figures / "03_library_size_normalization.png",
        dpi=180,
        bbox_inches="tight",
    )
    plt.show()


plot_library_size_normalization(raw_totals, normalized_totals, positive, FIGURES)

## 7 · Record processing metadata and save

Record the normalization parameters and HVG selection results, then save the normalized gene expression data as an AnnData file for downstream analysis.

In [ ]:
def save_normalized_data(adata, output_path, target_sum, n_top_genes, n_hvg):
    adata.uns["normalization_pipeline"] = {
        "method": "scanpy.pp.normalize_total + log1p",
        "target_sum": float(target_sum),
        "hvg_flavor": "seurat",
        "hvg_n_top_genes_requested": int(n_top_genes),
        "hvg_n_selected": int(n_hvg),
        "raw_count_layer": "counts",
        "input": "results/adata/02_qc.h5ad",
    }
    adata.write_h5ad(output_path)


save_normalized_data(adata, OUTPUT, TARGET_SUM, N_TOP_GENES, n_hvg)
print("Saved:", OUTPUT.relative_to(ROOT))
print("File size (MB):", round(OUTPUT.stat().st_size / (1024 ** 2), 2))

## 8 · Verify the saved AnnData file

Reload the saved AnnData file and verify that the normalized expression data, original raw counts, spatial coordinates, and HVG information are correctly preserved.

In [ ]:
def verify_normalized_output(output_path, adata, n_hvg):
    check = sc.read_h5ad(output_path)
    assert check.shape == adata.shape
    assert check.obs_names.equals(adata.obs_names)
    assert check.var_names.equals(adata.var_names)
    assert np.array_equal(check.obsm["spatial"], adata.obsm["spatial"])
    assert "counts" in check.layers and sparse.issparse(check.layers["counts"])
    assert np.array_equal(check.layers["counts"].data, adata.layers["counts"].data)
    assert "highly_variable" in check.var
    assert int(check.var["highly_variable"].sum()) == n_hvg
    assert np.isfinite(check.X.data).all()
    return check


check = verify_normalized_output(OUTPUT, adata, n_hvg)
print("PASS — 03_normalization completed")
print("AnnData:", check.shape)
print("HVGs:", n_hvg)
print("Raw count layer:", check.layers["counts"].shape)
print("Output:", OUTPUT.relative_to(ROOT))

## Next: 04. Clustering

In `04_clustering.ipynb`, we will perform PCA, construct a neighborhood graph, and apply UMAP and Leiden clustering to identify gene expression patterns across spatial bins.

**Input**: results/adata/03_normalized.h5ad

**Note**: Spatial bins represent tissue regions, not individual cells.
